## Music Recommendation Algorithm
This notebook shows the steps of how a music recommendation system works using Spotify audio features to identify songs with similar characteristics.

### Importing Libraries

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

### Data Loading

In [7]:
music_df = pd.read_csv("../data/spotify_dataset.csv")

print("Dataset shape:", music_df.shape)
music_df.head()

Dataset shape: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


### Data Cleaning
Unnecessary columns like unnamed, missing song information and duplicate records are removed.

In [8]:
music_df = music_df.drop(columns=["Unnamed: 0"])
music_df = music_df.dropna(subset=["track_name", "artists", "album_name"])
music_df = music_df.drop_duplicates()
music_df = music_df.reset_index(drop=True)

print(music_df.shape)

(113549, 20)


In [21]:
recommend_music_df = music_df.drop_duplicates(subset=["track_name", "artists"]).copy()
recommend_music_df = recommend_music_df.reset_index(drop=True)
print(recommend_music_df.shape)

(81343, 20)


### Audio Feature Selection
We selected 9 numerical audio features which includes danceability, energy, valence, acousticness, instrumentalness, speechiness, liveness, loudness and tempo. We did not include song names, artists and genres since they are categorical variables.

In [24]:
audio_features = ["danceability","energy","valence","acousticness", "instrumentalness","speechiness", "liveness",
                  "loudness", "tempo"]
display(recommend_music_df[audio_features].head())
print(recommend_music_df[audio_features].isnull().sum())

,danceability,energy,valence,acousticness,instrumentalness,speechiness,liveness,loudness,tempo
0,0.676,0.4610,0.715,0.0322,0.000001,0.1430,0.3580,-6.746,87.917
1,0.420,0.1660,0.267,0.9240,0.000006,0.0763,0.1010,-17.235,77.489
2,0.438,0.3590,0.120,0.2100,0.000000,0.0557,0.1170,-9.734,76.332
3,0.266,0.0596,0.143,0.9050,0.000071,0.0363,0.1320,-18.515,181.740
4,0.618,0.4430,0.167,0.4690,0.000000,0.0526,0.0829,-9.681,119.949


danceability        0
energy              0
valence             0
acousticness        0
instrumentalness    0
speechiness         0
liveness            0
loudness            0
tempo               0
dtype: int64


For 0 tempo values:

### Audio Feature Distributions and Filtering

In [27]:
display(recommend_music_df[audio_features].describe())

,danceability,energy,valence,acousticness,instrumentalness,speechiness,liveness,loudness,tempo
count,81343.000000,81343.000000,81343.000000,81343.000000,81343.000000,81343.000000,81343.000000,81343.000000,81343.000000
mean,0.559276,0.635025,0.463277,0.329666,0.184733,0.088993,0.219723,-8.593930,122.144834
std,0.177747,0.258641,0.263383,0.339961,0.331592,0.116629,0.198272,5.304797,30.129012
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-49.531000,0.000000
25%,0.446000,0.455000,0.241000,0.015900,0.000000,0.036100,0.098500,-10.451500,99.408000
50%,0.573000,0.678000,0.449000,0.190000,0.000089,0.049100,0.133000,-7.262000,122.030000
75%,0.690000,0.857000,0.676000,0.629000,0.153000,0.087000,0.283000,-5.140000,140.128500
max,0.985000,1.000000,0.995000,0.996000,1.000000,0.965000,1.000000,4.532000,243.372000


We can see that values are scaled differently. Tempo can reach 243 but danceability can only reach 1. Lets check songs with 0 tempo.

In [30]:
print(f"Songs with 0 tempo: {(recommend_music_df["tempo"] == 0).sum()}")

Songs with 0 tempo: 145


We can remove the songs with 0 tempo since it can mean these tempo are unavailable, and also it only accounts for 0.17% songs.

In [33]:
recommend_music_df = recommend_music_df[recommend_music_df["tempo"] > 0].copy()
recommend_music_df = recommend_music_df.reset_index(drop=True)

print(f"New shape: {recommend_music_df.shape}")

New shape: (81198, 20)


To address the previous issue of values being scaled differently, we are standardizing the audio features so that no features dominate in the similarity calculation.

In [36]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(recommend_music_df[audio_features])

print(scaled_features[:5])

[[ 0.65632204 -0.67872725  0.95438528 -0.87456318 -0.55574501  0.4615383
   0.70132226  0.34656571 -1.15941058]
 [-0.79554715 -1.82322576 -0.7497672   1.75051758 -0.55573126 -0.11015222
  -0.598369   -1.64746428 -1.51040486]
 [-0.6934626  -1.07445216 -1.30894223 -0.35119543 -0.55574806 -0.28671631
  -0.51745437 -0.22147334 -1.54934813]
 [-1.66893721 -2.23602115 -1.22145226  1.69458964 -0.5555344  -0.45299511
  -0.4415969  -1.89080095  1.99856188]
 [ 0.32738293 -0.74856106 -1.13015837  0.41119066 -0.55574806 -0.31328664
  -0.68990368 -0.21139768 -0.08125092]]


### Generating Music Recommendations
The recommend song function uses cosine similarity to compare a selected song with other songs based on the features. After that it returns the most similar songs.

In [39]:
def recommend_songs(song_index, n=4):
    """
    Recommend the top n most similar songs using cosine similarity.

    Parameters:
        song_index: Index of the selected song.
        n: Number of recommendations to return.
    """
    selected_song = scaled_features[song_index].reshape(1,-1)
    similarities = cosine_similarity(selected_song, scaled_features)[0]
    
    sorted_indices = np.argsort(similarities)[::-1]
    sorted_indices = sorted_indices[sorted_indices != song_index]
    
    top = sorted_indices[:n]
    results = recommend_music_df.iloc[top][["track_name","artists","track_genre"]].copy()
    results["similarity"] = similarities[top]
    return results

In [41]:
display(recommend_songs(100,10))

,track_name,artists,track_genre,similarity
32631,Oh Quão Cego Andei,Pr. Júlio Ribeiro,gospel,0.994716
518,Scared to Be Lonely,Boyce Avenue,acoustic,0.994249
233,Back For Good,The Mayries,acoustic,0.987139
13169,Inside Out,Mokita,chill,0.983528
16667,Better Boat,Kenny Chesney;Mindy Smith,country,0.981996
48593,Duhai Sayang,Yura Yunita,j-pop,0.981427
44911,Jet Skis On The Moat,Arctic Monkeys,indie,0.981251
53622,明天我要嫁給你,Wakin Chau,mandopop,0.981043
71459,Nayab,Nikhil Swaroop,singer-songwriter,0.980973
60570,Unna Nenachadhum,A.R. Rahman;Shreya Ghoshal;Sarthak Kalyani,pop-film,0.980843
